## Silver · `02_silver.fact_trips` and `02_silver.dq_exclusions`
Clean the raw trips with ordered data-quality rules.

- Rules run **in order**; every excluded row is counted under the **first** rule it fails.
- Each rule is backed by a check in `data_profile` (see the check number in its note).
- Reconciliation: raw rows = kept rows + sum of `rows_excluded`.

In [0]:
%run ../../config/config

In [0]:
%run ../../utils/zsi_functions

### Setup

In [0]:
from pyspark import StorageLevel
from pyspark.sql import functions as F

# Raw trips + helper columns from add_dq_columns():
#   _pu_nonexistent / _do_nonexistent : timestamp falls in the DST gap (local time that never existed)
#   _true_min                         : real elapsed minutes (DST-aware)  -> becomes duration_min
#   _trip_key                         : sha256 of all source columns      -> becomes trip_key
#   _dup_rn                           : 1 for the first copy of a row, 2+ for exact duplicates
raw = spark.table(tbl("01_bronze", "yellow_trips_raw"))
zones = spark.table(tbl("02_silver", "taxi_zone_lookup"))

trips = add_dq_columns(raw).persist(StorageLevel.MEMORY_AND_DISK)
raw_count = trips.count()

lookup_ids = [row.location_id for row in zones.select("location_id").collect()]

print(f"raw rows: {raw_count:,}")

In [0]:
# Column shortcuts used in the rules
pickup_ts       = F.col("tpep_pickup_datetime")
dropoff_ts      = F.col("tpep_dropoff_datetime")
pu_nonexistent  = F.col("_pu_nonexistent")
do_nonexistent  = F.col("_do_nonexistent")
true_min        = F.col("_true_min")
pu_location     = F.col("PULocationID")
do_location     = F.col("DOLocationID")
fare_amount     = F.col("fare_amount")
total_amount    = F.col("total_amount")

rules = {}   # rule_id -> (description, condition); order of definition = order of evaluation


def add_rule(rule_id, description, condition):
    """Register a rule and show how many raw rows match it on its own (before ordering)."""
    rules[rule_id] = (description, condition)
    display(trips.agg(F.sum(F.when(condition, 1).otherwise(0)).alias(f"rows_matching_{rule_id}")))

### A. Data-quality rules (in order)

#### Rule 1 · DUPLICATE_ROW
Rows identical in every source column (NULLs equal). One copy is kept. *Profile check 3.*

In [0]:
add_rule(
    "DUPLICATE_ROW",
    "Exact duplicate of another raw row (all columns equal, NULLs equal); one copy kept",
    F.col("_dup_rn") > 1,
)

#### Rule 2 · MISSING_TIMESTAMP
A trip without pickup or dropoff time cannot be placed in a window. *Profile check 2.*

In [0]:
add_rule(
    "MISSING_TIMESTAMP",
    "Pickup or dropoff timestamp is NULL",
    pickup_ts.isNull() | dropoff_ts.isNull(),
)

#### Rule 3 · NONEXISTENT_LOCAL_TIME
2025-03-09 02:00–02:59 never existed in New York. Such a timestamp means a wrong device clock,
so neither its window nor its duration can be trusted → excluded, not shifted. *Profile checks 6–7.*

In [0]:
add_rule(
    "NONEXISTENT_LOCAL_TIME",
    "Pickup or dropoff in the DST spring-forward gap (local time never existed)",
    pu_nonexistent | do_nonexistent,
)

#### Rule 4 · PICKUP_OUT_OF_PERIOD
Pickups outside March 2025 (wrong vendor clocks). The end is 2025-04-01 04:30 so that
operational day 2025-03-31 stays whole. *Profile checks 4–5.*

In [0]:
add_rule(
    "PICKUP_OUT_OF_PERIOD",
    f"Pickup outside [{PERIOD_START}, {PERIOD_END}) local time",
    (pickup_ts < F.to_timestamp(F.lit(PERIOD_START))) | (pickup_ts >= F.to_timestamp(F.lit(PERIOD_END))),
)

#### Rule 5 · NON_POSITIVE_DURATION
Dropoff at or before pickup, using the real (DST-aware) elapsed time. *Profile check 9.*

In [0]:
add_rule(
    "NON_POSITIVE_DURATION",
    "True (DST-aware) elapsed time <= 0 minutes",
    true_min <= 0,
)

#### Rule 6 · EXCESSIVE_DURATION
Longer than `MAX_DURATION_MIN` (config). Tune it to the tail shape in *profile checks 9–10*.

In [0]:
add_rule(
    "EXCESSIVE_DURATION",
    f"True elapsed time > {MAX_DURATION_MIN} minutes",
    true_min > MAX_DURATION_MIN,
)

#### Rule 7 · INVALID_LOCATION_ID
NULL or not in the lookup. IDs 264 / 265 are valid lookup rows, so those trips are kept
(they are only left out of the stress index). *Profile check 11.*

In [0]:
add_rule(
    "INVALID_LOCATION_ID",
    "PULocationID or DOLocationID NULL or not in zone lookup",
    F.coalesce(~pu_location.isin(lookup_ids), F.lit(True)) | F.coalesce(~do_location.isin(lookup_ids), F.lit(True)),
)

#### Rule 8 · NEGATIVE_AMOUNT
Negative fare or total = void / refund records, not trips. *Profile checks 12–13.*

In [0]:
add_rule(
    "NEGATIVE_AMOUNT",
    "fare_amount < 0 or total_amount < 0 (void / refund records)",
    (fare_amount < 0) | (total_amount < 0),
)

### B. Apply the rules — first failing rule wins

In [0]:
first_failed_rule = None
for rule_id, (_, condition) in rules.items():
    first_failed_rule = (F.when(condition, rule_id) if first_failed_rule is None
                         else first_failed_rule.when(condition, rule_id))

flagged = trips.withColumn("_rule", first_failed_rule).persist(StorageLevel.MEMORY_AND_DISK)

rows_per_rule = {row["_rule"]: row["rows"] for row in flagged.groupBy("_rule").agg(F.count("*").alias("rows")).collect()}
kept_count = rows_per_rule.pop(None, 0)

print(f"kept: {kept_count:,}   excluded: {raw_count - kept_count:,}")

### C. `02_silver.dq_exclusions`

In [0]:
dq_rows = [
    (order, rule_id, description, int(rows_per_rule.get(rule_id, 0)),
     round(100.0 * rows_per_rule.get(rule_id, 0) / raw_count, 4))
    for order, (rule_id, (description, _)) in enumerate(rules.items(), start=1)
]

excluded_count = sum(row[3] for row in dq_rows)
assert raw_count == kept_count + excluded_count, \
    f"reconciliation failed: raw {raw_count} != kept {kept_count} + excluded {excluded_count}"

dq_exclusions = spark.createDataFrame(
    dq_rows, "rule_order INT, rule_id STRING, description STRING, rows_excluded LONG, pct_of_raw DOUBLE"
)

dq_exclusions.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("02_silver", "dq_exclusions"))

display(dq_exclusions)

### D. `02_silver.fact_trips`

In [0]:
op_day_start_min = hhmm_to_min(ZSI_SETTINGS["spec"]["op_day_start"])

# every source column except the four that get new names -> all fare / fee columns unchanged
fee_columns = [c for c in raw.columns
               if c not in ("tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID",
                            "processing_timestamp")]

pickup_zones = zones.select(
    F.col("location_id").alias("pu_location_id"), F.col("zone").alias("pu_zone"), F.col("borough").alias("pu_borough")
)
dropoff_zones = zones.select(
    F.col("location_id").alias("do_location_id"), F.col("zone").alias("do_zone"), F.col("borough").alias("do_borough")
)

fact_trips = (
    flagged
    .filter(F.col("_rule").isNull())
    .select(
        F.col("_trip_key").alias("trip_key"),                    # unique + stable across reruns
        pickup_ts.alias("pickup_ts"),                            # as given (wall clock)
        dropoff_ts.alias("dropoff_ts"),
        true_min.alias("duration_min"),                          # real elapsed minutes
        pu_location.cast("int").alias("pu_location_id"),
        do_location.cast("int").alias("do_location_id"),
        F.to_date(pickup_ts - F.expr(f"INTERVAL {op_day_start_min} MINUTES")).alias("operational_date"),
        *fee_columns,
    )
    .join(F.broadcast(pickup_zones), "pu_location_id", "left")
    .join(F.broadcast(dropoff_zones), "do_location_id", "left")
    .select(
        "trip_key", "pickup_ts", "dropoff_ts", "duration_min",
        "pu_location_id", "do_location_id", "pu_zone", "pu_borough", "do_zone", "do_borough",
        "operational_date", *fee_columns,
    )
)

fact_trips.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("02_silver", "fact_trips"))

### E. Checks on the saved table

In [0]:
saved = spark.table(tbl("02_silver", "fact_trips"))

checks = saved.agg(
    F.count("*").alias("rows"),
    F.countDistinct("trip_key").alias("distinct_trip_keys"),
    F.sum(F.when(F.col("duration_min") <= 0, 1).otherwise(0)).alias("non_positive_durations"),
    F.min("operational_date").alias("first_operational_date"),
    F.max("operational_date").alias("last_operational_date"),
).first()

display(spark.createDataFrame([checks]))

assert checks.rows == kept_count, f"fact rows {checks.rows} != kept {kept_count}"
assert checks.distinct_trip_keys == checks.rows, "trip_key is not unique"
assert checks.non_positive_durations == 0, "found trips with duration_min <= 0"

In [0]:
flagged.unpersist()
trips.unpersist()
print(f"raw={raw_count:,}  kept={kept_count:,}  excluded={raw_count - kept_count:,}")